# Data Preparation Notebook

This notebook converts the raw OPSD file into `train_prices.csv` and `test_prices.csv`. Check `DATA_PATH` first, then run the cells in order.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "configs").exists():
    project_root = project_root.parent
if not (project_root / "configs").exists():
    raise RuntimeError("Could not locate the project root.")

data_dir = project_root / "data"
DATA_PATH = data_dir / "opsd_building.csv"
OUTPUT_DIR = data_dir
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH


In [2]:
def load_single_file(path):
    """Load one raw data file and normalize the column layout."""
    df = pd.read_csv(
        path,
        usecols=[1, 2, 3, 4],
        header=0,
        names=["load1", "load2", "load3", "price"],
        parse_dates=False,
    )
    start_time = pd.to_datetime("2020-01-01 00:00:00")
    df["timestamp"] = start_time + pd.to_timedelta(df.index * 15, "m")
    df.set_index("timestamp", inplace=True)
    assert len(df) % 96 == 0, "Daily records must be complete."
    print(f"Successfully loaded {len(df) / 96:.1f} days of data")
    return df

raw_df = load_single_file(DATA_PATH)
display(raw_df.head())


Successfully loaded 661.0 days of data


,load1,load2,load3,price
timestamp,,,,
2020-01-01 00:00:00,0.079,0.187,0.039,29.93
2020-01-01 00:15:00,0.078,0.000,0.031,29.93
2020-01-01 00:30:00,0.080,0.000,0.075,29.62
2020-01-01 00:45:00,0.110,0.000,0.176,29.62
2020-01-01 01:00:00,0.107,0.135,0.541,29.62


In [3]:
# Use the first 500 days for training and the remaining days for testing.
train_df = raw_df.iloc[: 500 * 96]
test_df = raw_df.iloc[500 * 96 :]

train_path = OUTPUT_DIR / "train_prices.csv"
test_path = OUTPUT_DIR / "test_prices.csv"
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print("Saved train data to", train_path)
print("Saved test data to", test_path)
